In [1]:
import torch
from diffusers import AutoPipelineForText2Image
from diffusers.utils import load_image
import os
os.environ['http_proxy'] = 'http://127.0.0.1:7896'
os.environ['https_proxy'] = 'http://127.0.0.1:7896'
os.environ['all_proxy'] = 'socks5://127.0.0.1:7897'

/home/wenxiao/anaconda3/envs/BCI/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@torch.no_grad()
def encode_image_manual(model, pixel_values):
    """
    手动展开 encode_image() 的 ConvStem + PosEmbed 部分，
    Transformer 不拆，保持与 open_clip 完全一致。
    """

    visual = model.visual
    x = pixel_values

    # -------------------------------------------
    # 1. ConvStem (ViT-H/14 的 patch embedding)
    # -------------------------------------------
    x = visual.conv1(x)  # [B, C, H/14, W/14]

    B, C, H, W = x.shape
    x = x.reshape(B, C, H * W).permute(0, 2, 1)  # [B, HW, C]

    # -------------------------------------------
    # 2. 添加 class token
    # -------------------------------------------
    cls_token = visual.class_embedding.to(x.dtype)
    cls_token = cls_token.unsqueeze(0).unsqueeze(0)  # [1,1,C]
    cls_token = cls_token.repeat(B, 1, 1)            # [B,1,C]

    x = torch.cat([cls_token, x], dim=1)  # [B, 1+HW, C]

    # -------------------------------------------
    # 3. 加 positional embedding
    # -------------------------------------------
    pos_embed = visual.positional_embedding.to(x.dtype)
    x = x + pos_embed

    # -------------------------------------------
    # 4. 前置 LayerNorm
    # -------------------------------------------
    x = visual.ln_pre(x)

    # -------------------------------------------
    # 5. Transformer（不拆）
    # -------------------------------------------
    x = visual.transformer(x)  # 自带 MHA + MLP 等所有流程

    # -------------------------------------------
    # 6. 最终 LayerNorm
    # -------------------------------------------
    x = visual.ln_post(x)

    # -------------------------------------------
    # 7. 只取 CLS token
    # -------------------------------------------
    image_feat = x[:, 0, :]  # [B, C]

    # -------------------------------------------
    # 8. projection
    # -------------------------------------------
    if visual.proj is not None:
        image_feat = image_feat @ visual.proj

    return image_feat

In [3]:
import torch
from PIL import Image
import open_clip
from model.custom_pipeline import *   # ← 根据你自己的路径修改


# --------------------------------------------------------------------------- #
# 1. 加载 open_clip 模型和 transforms
# --------------------------------------------------------------------------- #
device = "cuda:0" if torch.cuda.is_available() else "cpu"

model, _, preprocess = open_clip.create_model_and_transforms(
    model_name="ViT-H-14",
    pretrained="laion2b_s32b_b79k",
)
model = model.to(device)
model.eval()


# --------------------------------------------------------------------------- #
# 2. 读取图像并得到 pixel_values（用于提取 embedding）
# --------------------------------------------------------------------------- #
image_path = "/home/wenxiao/workspace/qhy/BMCA/image.png"
image = Image.open(image_path).convert("RGB")

pixel_values = preprocess(image).unsqueeze(0).to(device)  # [1,3,224,224]


# --------------------------------------------------------------------------- #
# 3. 提取图像特征（image_embeds）
# --------------------------------------------------------------------------- #
with torch.no_grad():
    # image_feature = model.encode_image(pixel_values)  # [1, 1024]
    image_feature = encode_image_manual(model, pixel_values)
    image_feature = image_feature / image_feature.norm(dim=-1, keepdim=True)  # 归一化
    image_feature = image_feature.unsqueeze(0)  # → [1,1,1024] 如果你的生成器需要该形状

print("Image feature shape:", image_feature.shape)


# --------------------------------------------------------------------------- #
# 4. 使用你的 Generator4Embeds 生成图像
# --------------------------------------------------------------------------- #
gen_device = "cuda:0" if torch.cuda.is_available() else "cpu"
generator = Generator4Embeds(device=gen_device)

generated = generator.generate(
    image_embeds=image_feature.to(gen_device),
    text_prompt="",
    generator=None
)

# --------------------------------------------------------------------------- #
# 5. 保存结果
# --------------------------------------------------------------------------- #
generated.save("generated.png")
print("图像已保存为 generated.png")

Image feature shape: torch.Size([1, 1, 1024])


  0%|          | 0/1 [00:00<?, ?it/s]/home/wenxiao/anaconda3/envs/BCI/lib/python3.10/site-packages/diffusers/models/embeddings.py:2587: FutureWarning: You have passed a tensor as `image_embeds`.This is deprecated and will be removed in a future release. Please make sure to update your script to pass `image_embeds` as a list of tensors to suppress this warning.
  deprecate("image_embeds not a list", "1.0.0", deprecation_message, standard_warn=False)
100%|██████████| 1/1 [00:00<00:00,  1.64it/s]


图像已保存为 generated.png
